# Day 1: Environment Setup & Data Exploration
# Smart City IoT Analytics Pipeline

---

## 🎯 LEARNING OBJECTIVES:
- Configure Spark cluster and development environment
- Understand IoT data characteristics and challenges  
- Implement basic data ingestion patterns
- Explore PySpark DataFrame operations

## 📅 SCHEDULE:
**Morning (4 hours):**
1. Environment Setup (2 hours)
2. Data Exploration (2 hours)

**Afternoon (4 hours):**  
3. Basic Data Ingestion (2 hours)
4. Initial Data Transformations (2 hours)

## ✅ DELIVERABLES:
- Working Spark cluster with all services running
- Data ingestion notebook with basic EDA
- Documentation of data quality findings  
- Initial data loading pipeline functions

---

In [6]:
print("🚀 Welcome to the Smart City IoT Analytics Pipeline!")
print("=" * 60)

🚀 Welcome to the Smart City IoT Analytics Pipeline!


In [7]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

# Import PySpark libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pyspark.sql.functions as F

---

# SECTION 1: ENVIRONMENT SETUP (Morning - 2 hours)

---

## TODO 1.1: Initialize Spark Session (15 minutes)

🎯 **TASK:** Create a Spark session configured for local development  
💡 **HINT:** Use SparkSession.builder with appropriate configurations  
📚 **DOCS:** https://spark.apache.org/docs/latest/sql-getting-started.html

**TODO:** Create Spark session with the following configurations:
- App name: "SmartCityIoTPipeline-Day1"
- Master: "local[*]" (use all available cores)
- Memory: "4g" for driver
- Additional configs for better performance

In [8]:
# TODO: Create Spark session with the following configurations:
spark = (SparkSession.builder
         .appName("SmartCityAirQuality")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.eventLog.enabled", "false")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
         .getOrCreate())

print("✅ Spark Session Details:")
print(f"   App Name: {spark.sparkContext.appName}")
print(f"   Spark Version: {spark.version}")
print(f"   Master: {spark.sparkContext.master}")
print(f"   Default Parallelism: {spark.sparkContext.defaultParallelism}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/12 18:09:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark Session Details:
   App Name: SmartCityAirQuality
   Spark Version: 4.2.0
   Master: local[*]
   Default Parallelism: 6


## TODO 1.2: Verify Infrastructure (15 minutes)

🎯 **TASK:** Check that all infrastructure services are running  
💡 **HINT:** Test database connectivity and file system access

In [9]:
from sparkcityx.database import connect_database
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../secrets/.env"))

def test_database_connection():
    """Test connection to the shared SparkCity PostgreSQL database."""
    try:
        with connect_database() as connection:
            with connection.cursor() as cursor:
                cursor.execute("SELECT current_database(), version();")
                database_name, server_version = cursor.fetchone()

        print("✅ Database connection successful!")
        print(f"   Database: {database_name}")
        print(f"   Server: {server_version.split(',')[0]}")
        return True

    except Exception as e:
        print(f"❌ Database connection failed: {e}")
        return False

db_connected = test_database_connection()

✅ Database connection successful!
   Database: smartcity_db
   Server: PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on x86_64-pc-linux-gnu


## TODO 1.3: Generate Sample Data (30 minutes)

🎯 **TASK:** Run the data generation script to create sample IoT data  
💡 **HINT:** Use the provided data generation script or run it manually

In [9]:
import os

def verify_sample_data():
    """Verify the expected Smart City datasets exist locally."""
    data_dir = "../data/raw"

    expected_files = [
        "city_zones.csv",
        "traffic_sensors.csv",
        "energy_meters.csv",
        "air_quality.json",
        "weather_data.parquet",
        "occupancy_data.csv",
        "fiscal_data.csv",
    ]

    missing_files = [
        filename
        for filename in expected_files
        if not os.path.exists(os.path.join(data_dir, filename))
    ]

    if missing_files:
        print("❌ Missing local datasets:")
        for filename in missing_files:
            print(f"   - {filename}")
        print("\nGenerate them with:")
        print("uv run python scripts/generate-data.py --records 36000")
        return False

    print("✅ All 7 local datasets are available.")
    return True

data_ready = verify_sample_data()

✅ All 7 local datasets are available.


---

# SECTION 2: DATA EXPLORATION (Morning - 2 hours)

---

In [10]:
print("\n" + "=" * 60)
print("📊 SECTION 2: EXPLORATORY DATA ANALYSIS")
print("=" * 60)


📊 SECTION 2: EXPLORATORY DATA ANALYSIS


## TODO 2.1: Load and Examine Data Sources (45 minutes)

🎯 **TASK:** Load each data source and examine its structure  
💡 **HINT:** Use appropriate Spark readers for different file formats  
📚 **CONCEPTS:** Schema inference, file formats, data types

In [12]:
# Define data directory
data_dir = "../data/raw"

In [13]:
# TODO: Load city zones reference data
print("📍 Loading City Zones Reference Data...")
try:
    zones_df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{data_dir}/city_zones.csv")
    
    # TODO: Display basic information about zones
    print(f"   📊 Records: {zones_df.count()}")
    print(f"   📋 Schema:")
    zones_df.printSchema()
    
    # TODO: Show sample data
    print(f"   🔍 Sample Data:")
    zones_df.show(5, truncate=False)
    
except Exception as e:
    print(f"❌ Error loading zones data: {str(e)}")

📍 Loading City Zones Reference Data...
   📊 Records: 36000
   📋 Schema:
root
 |-- zone_id: string (nullable = true)
 |-- zone_name: string (nullable = true)
 |-- zone_type: string (nullable = true)
 |-- lat_min: double (nullable = true)
 |-- lat_max: double (nullable = true)
 |-- lon_min: double (nullable = true)
 |-- lon_max: double (nullable = true)
 |-- population: integer (nullable = true)

   🔍 Sample Data:
+---------+---------------------+-----------+-------+---------+----------+----------+----------+
|zone_id  |zone_name            |zone_type  |lat_min|lat_max  |lon_min   |lon_max   |population|
+---------+---------------------+-----------+-------+---------+----------+----------+----------+
|ZONE-0001|Residential Zone 0001|residential|40.68  |40.680842|-74.05    |-74.049105|3824      |
|ZONE-0002|Commercial Zone 0002 |commercial |40.68  |40.680842|-74.049105|-74.048211|602       |
|ZONE-0003|Industrial Zone 0003 |industrial |40.68  |40.680842|-74.048211|-74.047316|663       |
|Z

In [15]:
# TODO: Load traffic sensors data  
print("\n🚗 Loading Traffic Sensors Data...")
try:
    # TODO: Load CSV file with proper options
    traffic_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{data_dir}/traffic_sensors.csv")
)
    
    # TODO: Display basic information
    print(f"   📊 Records: {traffic_df.count()}")
    print(f"   📋 Schema:")
    traffic_df.printSchema()
    
    # TODO: Show sample data
    print(f"   🔍 Sample Data:")
    traffic_df.show(5)
    
except Exception as e:
    print(f"❌ Error loading traffic data: {str(e)}")


🚗 Loading Traffic Sensors Data...
   📊 Records: 36000
   📋 Schema:
root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- location_lat: double (nullable = true)
 |-- location_lon: double (nullable = true)
 |-- vehicle_count: integer (nullable = true)
 |-- avg_speed: double (nullable = true)
 |-- congestion_level: string (nullable = true)
 |-- road_type: string (nullable = true)

   🔍 Sample Data:
+---------+-------------------+------------+------------+-------------+---------+----------------+-----------+
|sensor_id|          timestamp|location_lat|location_lon|vehicle_count|avg_speed|congestion_level|  road_type|
+---------+-------------------+------------+------------+-------------+---------+----------------+-----------+
| TRF-0001|2025-01-01 00:00:00|   40.680561|    -74.0492|          109|    16.17|            high|   arterial|
| TRF-0002|2025-01-01 00:05:00|   40.680511|  -74.049047|           81|    31.34|          medium|    highway|
| TR

In [16]:
# TODO: Load air quality data (JSON format)
print("\n🌫️ Loading Air Quality Data...")
try:
    # TODO: Load JSON file - note different file format!
    air_quality_df = spark.read.json(f"{data_dir}/air_quality.json")
    
    # TODO: Display basic information
    print(f"   📊 Records: {air_quality_df.count()}")
    print(f"   📋 Schema:")
    air_quality_df.printSchema()
    
    # TODO: Show sample data
    print(f"   🔍 Sample Data:")
    air_quality_df.show(5)
    
except Exception as e:
    print(f"❌ Error loading air quality data: {str(e)}")


🌫️ Loading Air Quality Data...
   📊 Records: 36000
   📋 Schema:
root
 |-- co: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- location_lat: double (nullable = true)
 |-- location_lon: double (nullable = true)
 |-- no2: double (nullable = true)
 |-- pm10: double (nullable = true)
 |-- pm25: double (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- timestamp: string (nullable = true)

   🔍 Sample Data:
+-----+--------+------------+------------+-----+-----+-----+---------+-----------+-------------------+
|   co|humidity|location_lat|location_lon|  no2| pm10| pm25|sensor_id|temperature|          timestamp|
+-----+--------+------------+------------+-----+-----+-----+---------+-----------+-------------------+
|0.621|   70.27|   40.680182|  -74.049643| 9.24|32.22|18.82| AIR-0001|      47.69|2025-01-01 00:00:00|
|1.305|   65.02|   40.680809|  -74.047195|16.88|26.64|22.63| AIR-0002|      50.99|2025-01-01 00:15:00|
|0

In [17]:
# TODO: Load weather data (Parquet format)
print("\n🌤️ Loading Weather Data...")
try:
    # TODO: Load Parquet file - another different format!
    weather_df = spark.read.parquet(f"{data_dir}/weather_data.parquet")
    
    # TODO: Display basic information
    print(f"   📊 Records: {weather_df.count()}")
    print(f"   📋 Schema:")
    weather_df.printSchema()
    
    # TODO: Show sample data
    print(f"   🔍 Sample Data:")
    weather_df.show(5)
    
except Exception as e:
    print(f"❌ Error loading weather data: {str(e)}")


🌤️ Loading Weather Data...
   📊 Records: 36000
   📋 Schema:
root
 |-- station_id: string (nullable = true)
 |-- timestamp: timestamp_ntz (nullable = true)
 |-- location_lat: double (nullable = true)
 |-- location_lon: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- wind_direction: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- pressure: double (nullable = true)

   🔍 Sample Data:
+----------+-------------------+------------+------------+-----------+--------+----------+--------------+-------------+--------+
|station_id|          timestamp|location_lat|location_lon|temperature|humidity|wind_speed|wind_direction|precipitation|pressure|
+----------+-------------------+------------+------------+-----------+--------+----------+--------------+-------------+--------+
|  WTH-0001|2025-01-01 00:00:00|   40.680459|  -74.049119|      51.69|   62.77|     22.56|    

In [18]:
# TODO: Load energy meters data
print("\n⚡ Loading Energy Meters Data...")
try:
    # TODO: Load CSV file
    energy_df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{data_dir}/energy_meters.csv")
    
    # TODO: Display basic information
    print(f"   📊 Records: {energy_df.count()}")
    print(f"   📋 Schema:")
    energy_df.printSchema()
    
    # TODO: Show sample data
    print(f"   🔍 Sample Data:")
    energy_df.show(5)
    
except Exception as e:
    print(f"❌ Error loading energy data: {str(e)}")


⚡ Loading Energy Meters Data...
   📊 Records: 36000
   📋 Schema:
root
 |-- meter_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- building_type: string (nullable = true)
 |-- location_lat: double (nullable = true)
 |-- location_lon: double (nullable = true)
 |-- power_consumption: double (nullable = true)
 |-- voltage: double (nullable = true)
 |-- current: double (nullable = true)
 |-- power_factor: double (nullable = true)

   🔍 Sample Data:
+--------+-------------------+-------------+------------+------------+-----------------+-------+-------+------------+
|meter_id|          timestamp|building_type|location_lat|location_lon|power_consumption|voltage|current|power_factor|
+--------+-------------------+-------------+------------+------------+-----------------+-------+-------+------------+
|ENG-0001|2025-01-01 00:00:00|   commercial|   40.680371|  -74.049869|             46.3| 120.77|  38.34|        0.92|
|ENG-0002|2025-01-01 00:10:00|   commercial|   40.

In [19]:
# TODO: Load occupancy data
print("\n🏢 Loading Occupancy Data...")
try:
    occupancy_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{data_dir}/occupancy_data.csv")
    )

    print(f"   📊 Records: {occupancy_df.count()}")
    print("   📋 Schema:")
    occupancy_df.printSchema()

    print("   🔍 Sample Data:")
    occupancy_df.show(5)

except Exception as e:
    print(f"❌ Error loading occupancy data: {str(e)}")


🏢 Loading Occupancy Data...
   📊 Records: 36000
   📋 Schema:
root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- location_lat: double (nullable = true)
 |-- location_lon: double (nullable = true)
 |-- available_rooms: integer (nullable = true)
 |-- occupied_rooms: integer (nullable = true)
 |-- guests: integer (nullable = true)

   🔍 Sample Data:
+---------+-------------------+------------+------------+---------------+--------------+------+
|sensor_id|          timestamp|location_lat|location_lon|available_rooms|occupied_rooms|guests|
+---------+-------------------+------------+------------+---------------+--------------+------+
| OCC-0001|2025-01-01 00:00:00|   40.680661|  -74.049734|            235|           130|   485|
| OCC-0002|2025-01-01 00:15:00|   40.680237|  -74.046201|             28|            10|    19|
| OCC-0003|2025-01-01 00:30:00|   40.680661|  -74.041989|           1227|           606|   831|
| OCC-0004|2025-01-01 00:45:00|

In [20]:
# TODO: Load fiscal data
print("\n💰 Loading Fiscal Data...")
try:
    fiscal_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{data_dir}/fiscal_data.csv")
    )

    print(f"   📊 Records: {fiscal_df.count()}")
    print("   📋 Schema:")
    fiscal_df.printSchema()

    print("   🔍 Sample Data:")
    fiscal_df.show(5)

except Exception as e:
    print(f"❌ Error loading fiscal data: {str(e)}")


💰 Loading Fiscal Data...
   📊 Records: 36000
   📋 Schema:
root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- location_lat: double (nullable = true)
 |-- location_lon: double (nullable = true)
 |-- expense: double (nullable = true)
 |-- revenue: double (nullable = true)

   🔍 Sample Data:
+---------+-------------------+------------+------------+-------+-------+
|sensor_id|          timestamp|location_lat|location_lon|expense|revenue|
+---------+-------------------+------------+------------+-------+-------+
| FIS-0001|2025-01-01 00:00:00|   40.680301|  -74.049417|  11.58| 134.59|
| FIS-0002|2025-01-01 00:15:00|   40.680048|  -74.046101|   6.17|  28.82|
| FIS-0003|2025-01-01 00:30:00|   40.680603|  -74.042012| 267.68| 966.66|
| FIS-0004|2025-01-01 00:45:00|   40.680237|  -74.038751|  24.55| 160.53|
| FIS-0005|2025-01-01 01:00:00|   40.680046|  -74.035404|   0.96|   8.71|
+---------+-------------------+------------+------------+-------+-------+


## TODO 2.2: Basic Data Quality Assessment (45 minutes)

🎯 **TASK:** Assess data quality across all datasets  
💡 **HINT:** Check for missing values, duplicates, data ranges  
📚 **CONCEPTS:** Data profiling, quality metrics, anomaly detection

In [22]:
def assess_data_quality(df, dataset_name):
    """
    Perform basic data quality assessment on a DataFrame
    
    Args:
        df: Spark DataFrame to assess
        dataset_name: Name of the dataset for reporting
    """
    print(f"\n📋 Data Quality Assessment: {dataset_name}")
    print("-" * 50)
    
    # TODO: Basic statistics
    total_rows = df.count()
    total_cols = len(df.columns)
    print(f"   📊 Dimensions: {total_rows:,} rows × {total_cols} columns")
    
    # TODO: Check for missing values
    print(f"   🔍 Missing Values:")
    for col in df.columns:
        missing_count = df.filter(F.col(col).isNull()).count()
        missing_pct = (missing_count / total_rows) * 100
        if missing_count > 0:
            print(f"      {col}: {missing_count:,} ({missing_pct:.2f}%)")
    
    # TODO: Check for duplicate records
    duplicate_count = total_rows - df.dropDuplicates().count()
    if duplicate_count > 0:
        print(f"   🔄 Duplicate Records: {duplicate_count:,}")
    else:
        print(f"   ✅ No duplicate records found")
    
    # TODO: Numeric column statistics
    numeric_cols = [field.name for field in df.schema.fields 
                   if field.dataType in [IntegerType(), DoubleType(), FloatType(), LongType()]]
    
    if numeric_cols:
        print(f"   📈 Numeric Columns Summary:")
        # Show basic statistics for numeric columns
        df.select(numeric_cols).describe().show()

In [23]:
# TODO: Assess quality for each dataset
datasets = [
    (zones_df, "City Zones"),
    (traffic_df, "Traffic Sensors"), 
    (air_quality_df, "Air Quality"),
    (weather_df, "Weather Stations"),
    (energy_df, "Energy Meters"),
    (occupancy_df, "Occupancy"),
    (fiscal_df, "Fiscal")
]

for df, name in datasets:
    try:
        assess_data_quality(df, name)
    except Exception as e:
        print(f"❌ Error assessing {name}: {str(e)}")


📋 Data Quality Assessment: City Zones
--------------------------------------------------
   📊 Dimensions: 36,000 rows × 8 columns
   🔍 Missing Values:
   ✅ No duplicate records found
   📈 Numeric Columns Summary:


26/09/12 18:23:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------------------+--------------------+--------------------+--------------------+------------------+
|summary|            lat_min|             lat_max|             lon_min|             lon_max|        population|
+-------+-------------------+--------------------+--------------------+--------------------+------------------+
|  count|              36000|               36000|               36000|               36000|             36000|
|   mean|  40.75935789444529|  40.760200000001056|  -73.96555921051029|  -73.96466447367668| 3022.003583333333|
| stddev|0.04606103919488944|0.046061039701610976|0.049078050635065436|0.049078050635950575|2855.8303511974086|
|    min|              40.68|           40.680842|              -74.05|          -74.049105|                 0|
|    max|          40.839158|               40.84|          -73.880895|              -73.88|             12000|
+-------+-------------------+--------------------+--------------------+--------------------+------------

## TODO 2.3: Temporal Analysis (30 minutes)

🎯 **TASK:** Analyze temporal patterns in the IoT data  
💡 **HINT:** Look at data distribution over time, identify patterns  
📚 **CONCEPTS:** Time series analysis, temporal patterns, data distribution

In [24]:
print("\n" + "=" * 60) 
print("⏰ TEMPORAL PATTERN ANALYSIS")
print("=" * 60)


⏰ TEMPORAL PATTERN ANALYSIS


In [28]:
# TODO: Analyze traffic patterns by hour
print("\n🚗 Traffic Patterns by Hour:")
try:
    # TODO: Extract hour from timestamp and analyze vehicle counts
    traffic_hourly = (traffic_df
                     .withColumn("hour", F.hour("timestamp"))
                     .groupBy("hour")
                     .agg(F.avg("vehicle_count").alias("avg_vehicles"),
                          F.count("*").alias("readings"))
                     .orderBy("hour"))
    
    # TODO: Show the results
    traffic_hourly.show(24)
    
    # TODO: What patterns do you notice? Add your observations here:
    print("📝 OBSERVATIONS:")
    print("   - Rush hour patterns: Traffic shows clear morning and evening rush-hour patterns, with higher vehicle counts around 7–8 AM and 4–6 PM.")
    print("   - Off-peak periods: Traffic remains relatively steady during off-peak hours, generally averaging about 67–71 vehicles.")
    print("   - Peak traffic hours: The highest average vehicle count occurs at 5 PM (hour 17), with approximately 107.6 vehicles.")
    
except Exception as e:
    print(f"❌ Error analyzing traffic patterns: {str(e)}")


🚗 Traffic Patterns by Hour:
+----+------------------+--------+
|hour|      avg_vehicles|readings|
+----+------------------+--------+
|   0| 69.52133333333333|    1500|
|   1| 68.15933333333334|    1500|
|   2| 68.19825268817205|    1488|
|   3| 68.20833333333333|    1512|
|   4|             68.82|    1500|
|   5| 66.65733333333333|    1500|
|   6| 68.08333333333333|    1500|
|   7|104.25333333333333|    1500|
|   8|105.98066666666666|    1500|
|   9| 69.00666666666666|    1500|
|  10|            68.844|    1500|
|  11| 68.98866666666666|    1500|
|  12| 69.62666666666667|    1500|
|  13|             68.74|    1500|
|  14|            67.744|    1500|
|  15| 67.19533333333334|    1500|
|  16|           106.646|    1500|
|  17|107.59666666666666|    1500|
|  18|106.16266666666667|    1500|
|  19|            71.422|    1500|
|  20| 69.54466666666667|    1500|
|  21| 67.12866666666666|    1500|
|  22| 69.45933333333333|    1500|
|  23| 67.58933333333333|    1500|
+----+------------------+-

In [27]:
# TODO: Analyze air quality patterns by day of week
air_quality_df = air_quality_df.withColumn(
    "timestamp",
    F.to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss")
)

print("\n🌫️ Air Quality Patterns by Day of Week:")
try:
    # TODO: Extract day of week and analyze PM2.5 levels
    air_quality_daily = (air_quality_df
                        .withColumn("day_of_week", F.dayofweek("timestamp"))
                        .groupBy("day_of_week")
                        .agg(F.avg("pm25").alias("avg_pm25"),
                             F.avg("no2").alias("avg_no2"))
                        .orderBy("day_of_week"))
    
    # TODO: Show results
    air_quality_daily.show()
    
    # TODO: Add your observations
    print("📝 OBSERVATIONS:")
    print("   - Weekday vs weekend patterns: Air quality levels remain fairly consistent across weekdays and weekends, with no significant difference.")
    print("   - Pollution trends: PM2.5 averages stay near 21 and NO2 averages stay near 26.5 throughout the week, showing little day-to-day variation.")
    
except Exception as e:
    print(f"❌ Error analyzing air quality patterns: {str(e)}")


🌫️ Air Quality Patterns by Day of Week:
+-----------+------------------+------------------+
|day_of_week|          avg_pm25|           avg_no2|
+-----------+------------------+------------------+
|          1|21.119834905660387| 26.60579795597484|
|          2|20.950420597484243|26.584455581760977|
|          3|20.956096698113214|26.644488993710674|
|          4| 20.96931134259261|26.567878086419736|
|          5|21.023661265432104|26.742860725308653|
|          6|21.210738811728405|26.355862268518482|
|          7| 21.01877121913577|26.453763503086453|
+-----------+------------------+------------------+

📝 OBSERVATIONS:
   - Weekday vs weekend patterns: Air quality levels remain fairly consistent across weekdays and weekends, with no significant difference.
   - Pollution trends: PM2.5 averages stay near 21 and NO2 averages stay near 26.5 throughout the week, showing little day-to-day variation.


---

# SECTION 3: BASIC DATA INGESTION (Afternoon - 2 hours)

---

In [29]:
print("\n" + "=" * 60)
print("📥 SECTION 3: DATA INGESTION PIPELINE")
print("=" * 60)


📥 SECTION 3: DATA INGESTION PIPELINE


## TODO 3.1: Create Reusable Data Loading Functions (60 minutes)

🎯 **TASK:** Create reusable functions for loading different data formats  
💡 **HINT:** Handle schema validation and error handling  
📚 **CONCEPTS:** Function design, error handling, schema enforcement

In [31]:
def load_csv_data(file_path, expected_schema=None):
    """
    Load CSV data with proper error handling and schema validation
    """
    try:
        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(file_path)
        )

        if expected_schema:
            if df.schema != expected_schema:
                print(f"⚠️ Schema mismatch for {file_path}")
                print("Expected:")
                print(expected_schema.simpleString())
                print("Actual:")
                print(df.schema.simpleString())
                return None

        print(f"✅ Successfully loaded CSV: {file_path}")
        return df

    except Exception as e:
        print(f"❌ Error loading CSV {file_path}: {str(e)}")
        return None


def load_json_data(file_path):
    """
    Load JSON data with error handling
    """
    try:
        df = spark.read.json(file_path)

        print(f"✅ Successfully loaded JSON: {file_path}")
        return df

    except Exception as e:
        print(f"❌ Error loading JSON {file_path}: {str(e)}")
        return None


def load_parquet_data(file_path):
    """
    Load Parquet data with error handling
    """
    try:
        df = spark.read.parquet(file_path)

        print(f"✅ Successfully loaded Parquet: {file_path}")
        return df

    except Exception as e:
        print(f"❌ Error loading Parquet {file_path}: {str(e)}")
        return None

In [32]:
def assess_data_quality(df, dataset_name):
    """
    Perform basic data quality assessment on a DataFrame
    
    Args:
        df: Spark DataFrame to assess
        dataset_name: Name of the dataset for reporting
    """
    print(f"\n📋 Data Quality Assessment: {dataset_name}")
    print("-" * 50)
    
    # TODO: Basic statistics
    total_rows = df.count()
    total_cols = len(df.columns)
    print(f"   📊 Dimensions: {total_rows:,} rows × {total_cols} columns")
    
    # TODO: Check for missing values
    print(f"   🔍 Missing Values:")
    for col in df.columns:
        missing_count = df.filter(F.col(col).isNull()).count()
        missing_pct = (missing_count / total_rows) * 100
        if missing_count > 0:
            print(f"      {col}: {missing_count:,} ({missing_pct:.2f}%)")
    
    # TODO: Check for duplicate records
    duplicate_count = total_rows - df.dropDuplicates().count()
    if duplicate_count > 0:
        print(f"   🔄 Duplicate Records: {duplicate_count:,}")
    else:
        print(f"   ✅ No duplicate records found")
    
    # TODO: Numeric column statistics
    numeric_cols = [field.name for field in df.schema.fields 
                   if field.dataType in [IntegerType(), DoubleType(), FloatType(), LongType()]]
    
    if numeric_cols:
        print(f"   📈 Numeric Columns Summary:")
        # Show basic statistics for numeric columns
        df.select(numeric_cols).describe().show()

In [33]:
# TODO: Test your loading functions
print("🧪 Testing Data Loading Functions:")

test_files = [
    (f"{data_dir}/city_zones.csv", "CSV", load_csv_data),
    (f"{data_dir}/air_quality.json", "JSON", load_json_data), 
    (f"{data_dir}/weather_data.parquet", "Parquet", load_parquet_data)
]

for file_path, file_type, load_func in test_files:
    print(f"\n   Testing {file_type} loader...")
    test_df = load_func(file_path)
    if test_df:
        print(f"      Records loaded: {test_df.count():,}")

🧪 Testing Data Loading Functions:

   Testing CSV loader...
✅ Successfully loaded CSV: ../data/raw/city_zones.csv
      Records loaded: 36,000

   Testing JSON loader...
✅ Successfully loaded JSON: ../data/raw/air_quality.json
      Records loaded: 36,000

   Testing Parquet loader...
✅ Successfully loaded Parquet: ../data/raw/weather_data.parquet
      Records loaded: 36,000


## TODO 3.2: Schema Definition and Enforcement (60 minutes)

🎯 **TASK:** Define explicit schemas for data consistency  
💡 **HINT:** Use StructType and StructField for schema definition  
📚 **CONCEPTS:** Schema design, data types, schema enforcement

In [34]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType
)

# Traffic sensors schema
traffic_schema = StructType([
    StructField("sensor_id", StringType(), False),
    StructField("timestamp", TimestampType(), False),
    StructField("location_lat", DoubleType(), False),
    StructField("location_lon", DoubleType(), False),
    StructField("vehicle_count", IntegerType(), False),
    StructField("avg_speed", DoubleType(), False),
    StructField("congestion_level", StringType(), False),
    StructField("road_type", StringType(), False),
])


# Air quality schema
air_quality_schema = StructType([
    StructField("co", DoubleType(), False),
    StructField("humidity", DoubleType(), False),
    StructField("location_lat", DoubleType(), False),
    StructField("location_lon", DoubleType(), False),
    StructField("no2", DoubleType(), False),
    StructField("pm10", DoubleType(), False),
    StructField("pm25", DoubleType(), False),
    StructField("sensor_id", StringType(), False),
    StructField("temperature", DoubleType(), False),
    StructField("timestamp", TimestampType(), False),
])


# Weather schema
weather_schema = StructType([
    StructField("station_id", StringType(), False),
    StructField("timestamp", TimestampType(), False),
    StructField("location_lat", DoubleType(), False),
    StructField("location_lon", DoubleType(), False),
    StructField("temperature", DoubleType(), False),
    StructField("humidity", DoubleType(), False),
    StructField("wind_speed", DoubleType(), False),
    StructField("wind_direction", DoubleType(), False),
    StructField("precipitation", DoubleType(), False),
    StructField("pressure", DoubleType(), False),
])


# Energy schema
energy_schema = StructType([
    StructField("meter_id", StringType(), False),
    StructField("timestamp", TimestampType(), False),
    StructField("building_type", StringType(), False),
    StructField("location_lat", DoubleType(), False),
    StructField("location_lon", DoubleType(), False),
    StructField("power_consumption", DoubleType(), False),
    StructField("voltage", DoubleType(), False),
    StructField("current", DoubleType(), False),
    StructField("power_factor", DoubleType(), False),
])

In [35]:
# TODO: Test schema enforcement
print("\n🔍 Testing Schema Enforcement:")

def load_with_schema(file_path, schema, file_format="csv"):
    """Load data with explicit schema enforcement"""
    try:
        if file_format == "csv":
            df = spark.read.schema(schema).option("header", "true").csv(file_path)
        elif file_format == "json":
            df = spark.read.schema(schema).json(file_path)
        elif file_format == "parquet":
            df = spark.read.schema(schema).parquet(file_path)
        
        print(f"✅ Schema enforcement successful for {file_path}")
        return df
        
    except Exception as e:
        print(f"❌ Schema enforcement failed for {file_path}: {str(e)}")
        return None

# TODO: Test with one of your schemas
test_schema_df = load_with_schema(f"{data_dir}/traffic_sensors.csv", traffic_schema, "csv")
if test_schema_df:
    print("   Schema enforcement test passed!")
    test_schema_df.printSchema()


🔍 Testing Schema Enforcement:
✅ Schema enforcement successful for ../data/raw/traffic_sensors.csv
   Schema enforcement test passed!
root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- location_lat: double (nullable = true)
 |-- location_lon: double (nullable = true)
 |-- vehicle_count: integer (nullable = true)
 |-- avg_speed: double (nullable = true)
 |-- congestion_level: string (nullable = true)
 |-- road_type: string (nullable = true)



---

# SECTION 4: INITIAL DATA TRANSFORMATIONS (Afternoon - 2 hours)

---

In [36]:
print("\n" + "=" * 60)
print("🔄 SECTION 4: DATA TRANSFORMATIONS")
print("=" * 60)


🔄 SECTION 4: DATA TRANSFORMATIONS


## TODO 4.1: Timestamp Standardization (45 minutes)

🎯 **TASK:** Standardize timestamp formats across all datasets  
💡 **HINT:** Some datasets may have different timestamp formats  
📚 **CONCEPTS:** Date/time handling, format standardization, timezone handling

In [37]:
def standardize_timestamps(df, timestamp_col="timestamp"):
    """
    Standardize timestamp column across datasets
    
    Args:
        df: Input DataFrame
        timestamp_col: Name of timestamp column
        
    Returns:
        DataFrame with standardized timestamps
    """
    try:
        # TODO: Convert timestamps to standard format
        standardized_df = (df
                          .withColumn("timestamp_std", F.to_timestamp(F.col(timestamp_col)))
                          .drop(timestamp_col)
                          .withColumnRenamed("timestamp_std", timestamp_col))
        
        # TODO: Add derived time columns
        result_df = (standardized_df
                    .withColumn("year", F.year(timestamp_col))
                    .withColumn("month", F.month(timestamp_col))
                    .withColumn("day", F.dayofmonth(timestamp_col))
                    .withColumn("hour", F.hour(timestamp_col))
                    .withColumn("day_of_week", F.dayofweek(timestamp_col))
                    .withColumn("is_weekend", F.when(F.dayofweek(timestamp_col).isin([1, 7]), True).otherwise(False)))
        
        return result_df
        
    except Exception as e:
        print(f"❌ Error standardizing timestamps: {str(e)}")
        return df

In [38]:
# TODO: Test timestamp standardization
print("⏰ Testing Timestamp Standardization:")

# Test with traffic data
traffic_std = standardize_timestamps(traffic_df)
print("   Traffic data timestamp standardization:")
traffic_std.select("timestamp", "year", "month", "day", "hour", "day_of_week", "is_weekend").show(5)

⏰ Testing Timestamp Standardization:
   Traffic data timestamp standardization:
+-------------------+----+-----+---+----+-----------+----------+
|          timestamp|year|month|day|hour|day_of_week|is_weekend|
+-------------------+----+-----+---+----+-----------+----------+
|2025-01-01 00:00:00|2025|    1|  1|   0|          4|     false|
|2025-01-01 00:05:00|2025|    1|  1|   0|          4|     false|
|2025-01-01 00:10:00|2025|    1|  1|   0|          4|     false|
|2025-01-01 00:15:00|2025|    1|  1|   0|          4|     false|
|2025-01-01 00:20:00|2025|    1|  1|   0|          4|     false|
+-------------------+----+-----+---+----+-----------+----------+
only showing top 5 rows


## TODO 4.2: Geographic Zone Mapping (45 minutes)

🎯 **TASK:** Map sensor locations to city zones  
💡 **HINT:** Join sensor coordinates with zone boundaries  
📚 **CONCEPTS:** Spatial joins, geographic data, coordinate systems

In [39]:
def map_to_zones(sensor_df, zones_df):
    """
    Map sensor locations to city zones
    
    Args:
        sensor_df: DataFrame with sensor locations (lat, lon)
        zones_df: DataFrame with zone boundaries
        
    Returns:
        DataFrame with zone information added
    """
    try:
        # TODO: Create join condition for geographic mapping
        # A sensor is in a zone if its coordinates fall within zone boundaries
        join_condition = (
            (sensor_df.location_lat >= zones_df.lat_min) &
            (sensor_df.location_lat <= zones_df.lat_max) &
            (sensor_df.location_lon >= zones_df.lon_min) &
            (sensor_df.location_lon <= zones_df.lon_max)
        )
        
        # TODO: Perform the join
        result_df = (sensor_df
                    .join(zones_df, join_condition, "left")
                    .select(sensor_df["*"], 
                           zones_df.zone_id, 
                           zones_df.zone_name, 
                           zones_df.zone_type))
        
        return result_df
        
    except Exception as e:
        print(f"❌ Error mapping to zones: {str(e)}")
        return sensor_df

In [40]:
# TODO: Test zone mapping
print("\n🗺️ Testing Geographic Zone Mapping:")

# Test with traffic sensors
traffic_with_zones = map_to_zones(traffic_std, zones_df)
print("   Traffic sensors with zone mapping:")
traffic_with_zones.select("sensor_id", "location_lat", "location_lon", "zone_id", "zone_type").show(10)

# TODO: Verify mapping worked correctly
zone_distribution = traffic_with_zones.groupBy("zone_type").count().orderBy(F.desc("count"))
print("   Sensors by zone type:")
zone_distribution.show()


🗺️ Testing Geographic Zone Mapping:
   Traffic sensors with zone mapping:
+---------+------------+------------+---------+-----------+
|sensor_id|location_lat|location_lon|  zone_id|  zone_type|
+---------+------------+------------+---------+-----------+
| TRF-0001|   40.680561|    -74.0492|ZONE-0001|residential|
| TRF-0002|   40.680511|  -74.049047|ZONE-0002| commercial|
| TRF-0003|   40.680129|  -74.047605|ZONE-0003| industrial|
| TRF-0004|   40.680509|  -74.047168|ZONE-0004|  mixed_use|
| TRF-0005|   40.680773|  -74.045866|ZONE-0005|       park|
| TRF-0006|   40.680436|   -74.04495|ZONE-0006|     campus|
| TRF-0007|    40.68037|  -74.044243|ZONE-0007|residential|
| TRF-0008|   40.680756|  -74.043306|ZONE-0008| commercial|
| TRF-0009|   40.680222|  -74.042132|ZONE-0009| industrial|
| TRF-0010|   40.680187|  -74.041262|ZONE-0010|  mixed_use|
+---------+------------+------------+---------+-----------+
only showing top 10 rows
   Sensors by zone type:


+-----------+-----+
|  zone_type|count|
+-----------+-----+
|     campus| 6020|
| industrial| 6014|
| commercial| 6012|
|residential| 6012|
|  mixed_use| 6011|
|       park| 6010|
+-----------+-----+



## TODO 4.3: Data Type Conversions and Validations (30 minutes)

🎯 **TASK:** Ensure proper data types and add validation columns  
💡 **HINT:** Cast columns to appropriate types, add data quality flags  
📚 **CONCEPTS:** Data type conversion, validation rules, data quality flags

In [41]:
def add_data_quality_flags(df, sensor_type):
    """
    Add data quality validation flags to DataFrame
    
    Args:
        df: Input DataFrame
        sensor_type: Type of sensor for specific validations
        
    Returns:
        DataFrame with quality flags added
    """
    try:
        result_df = df

        # General quality flag
        result_df = result_df.withColumn(
            "has_missing_values",
            F.when(F.col("sensor_id").isNull(), True).otherwise(False)
        )

        # Traffic-specific validations
        if sensor_type == "traffic":
            result_df = (
                result_df
                .withColumn(
                    "valid_speed",
                    F.when(
                        (F.col("avg_speed") >= 0) &
                        (F.col("avg_speed") <= 100),
                        True
                    ).otherwise(False)
                )
                .withColumn(
                    "valid_vehicle_count",
                    F.when(
                        F.col("vehicle_count") >= 0,
                        True
                    ).otherwise(False)
                )
            )

        # Air-quality-specific validations
        elif sensor_type == "air_quality":
            result_df = (
                result_df
                .withColumn(
                    "valid_pm25",
                    F.when(
                        (F.col("pm25") >= 0) &
                        (F.col("pm25") <= 500),
                        True
                    ).otherwise(False)
                )
                .withColumn(
                    "valid_temperature",
                    F.when(
                        (F.col("temperature") >= -50) &
                        (F.col("temperature") <= 50),
                        True
                    ).otherwise(False)
                )
            )

        return result_df

    except Exception as e:
        print(f"❌ Error adding quality flags: {str(e)}")
        return df

In [42]:
# TODO: Test data quality flags
print("\n🏷️ Testing Data Quality Flags:")

# Test with traffic data
traffic_with_flags = add_data_quality_flags(traffic_with_zones, "traffic")
print("   Traffic data with quality flags:")
traffic_with_flags.select("sensor_id", "avg_speed", "vehicle_count", "valid_speed", "valid_vehicle_count").show(10)

# TODO: Check quality flag distribution
quality_stats = (traffic_with_flags
                .agg(F.sum(F.when(F.col("valid_speed"), 1).otherwise(0)).alias("valid_speed_count"),
                     F.sum(F.when(F.col("valid_vehicle_count"), 1).otherwise(0)).alias("valid_vehicle_count_count"),
                     F.count("*").alias("total_records")))

print("   Quality statistics:")
quality_stats.show()


🏷️ Testing Data Quality Flags:
   Traffic data with quality flags:
+---------+---------+-------------+-----------+-------------------+
|sensor_id|avg_speed|vehicle_count|valid_speed|valid_vehicle_count|
+---------+---------+-------------+-----------+-------------------+
| TRF-0001|    16.17|          109|       true|               true|
| TRF-0002|    31.34|           81|       true|               true|
| TRF-0003|     15.2|           55|       true|               true|
| TRF-0004|     33.1|           46|       true|               true|
| TRF-0005|    18.56|           42|       true|               true|
| TRF-0006|    23.17|          104|       true|               true|
| TRF-0007|    13.18|           68|       true|               true|
| TRF-0008|    37.14|           66|       true|               true|
| TRF-0009|    18.64|           46|       true|               true|
| TRF-0010|    29.07|           88|       true|               true|
+---------+---------+-------------+-----------+-

+-----------------+-------------------------+-------------+
|valid_speed_count|valid_vehicle_count_count|total_records|
+-----------------+-------------------------+-------------+
|            36079|                    36079|        36079|
+-----------------+-------------------------+-------------+



---

# DAY 1 DELIVERABLES & CHECKPOINTS

---

In [43]:
print("\n" + "=" * 60)
print("📋 DAY 1 COMPLETION CHECKLIST")
print("=" * 60)


📋 DAY 1 COMPLETION CHECKLIST


In [45]:
def validate_day1_completion():
    """Validate that Day 1 objectives have been met"""

    checklist = {
        "spark_session_created": False,
        "database_connection_tested": False,
        "data_loaded_successfully": False,
        "data_quality_assessed": False,
        "loading_functions_created": False,
        "schemas_defined": False,
        "timestamp_standardization_working": False,
        "zone_mapping_implemented": False,
        "quality_flags_added": False
    }

    try:
        # 1. Spark session created
        if spark and spark.sparkContext._jsc:
            checklist["spark_session_created"] = True

        # 2. Database connection tested
        if "db_connected" in globals() and db_connected:
            checklist["database_connection_tested"] = True

        # 3. Data loaded successfully
        if "traffic_df" in globals() and traffic_df.count() > 0:
            checklist["data_loaded_successfully"] = True

        # 4. Data quality assessed
        if "assess_data_quality" in globals():
            checklist["data_quality_assessed"] = True

        # 5. Reusable loading functions created
        if all(
            name in globals()
            for name in [
                "load_csv_data",
                "load_json_data",
                "load_parquet_data"
            ]
        ):
            checklist["loading_functions_created"] = True

        # 6. Schemas defined
        if all(
            name in globals()
            for name in [
                "traffic_schema",
                "air_quality_schema",
                "weather_schema",
                "energy_schema"
            ]
        ):
            checklist["schemas_defined"] = True

        # 7. Timestamp standardization working
        if (
            "traffic_std" in globals()
            and "timestamp" in traffic_std.columns
            and "year" in traffic_std.columns
            and "month" in traffic_std.columns
            and "day" in traffic_std.columns
            and "hour" in traffic_std.columns
            and "day_of_week" in traffic_std.columns
            and "is_weekend" in traffic_std.columns
        ):
            checklist["timestamp_standardization_working"] = True

        # 8. Zone mapping implemented
        if (
            "traffic_with_zones" in globals()
            and "zone_id" in traffic_with_zones.columns
            and "zone_type" in traffic_with_zones.columns
        ):
            checklist["zone_mapping_implemented"] = True

        # 9. Quality flags added
        if (
            "traffic_with_flags" in globals()
            and "valid_speed" in traffic_with_flags.columns
            and "valid_vehicle_count" in traffic_with_flags.columns
        ):
            checklist["quality_flags_added"] = True

    except Exception as e:
        print(f"❌ Validation error: {str(e)}")

    # Display results
    print("✅ COMPLETION STATUS:")

    for item, status in checklist.items():
        status_icon = "✅" if status else "❌"
        print(f"   {status_icon} {item.replace('_', ' ').title()}")

    import builtins
    
    completion_rate = builtins.sum(checklist.values()) / len(checklist) * 100

    print(f"\n📊 Overall Completion: {completion_rate:.1f}%")

    if completion_rate >= 80:
        print("🎉 Great job! You're ready for Day 2!")
    else:
        print("📝 Please review incomplete items before proceeding to Day 2.")

    return checklist


# Run the validation
completion_status = validate_day1_completion()

✅ COMPLETION STATUS:
   ✅ Spark Session Created
   ✅ Database Connection Tested
   ✅ Data Loaded Successfully
   ✅ Data Quality Assessed
   ✅ Loading Functions Created
   ✅ Schemas Defined
   ✅ Timestamp Standardization Working
   ✅ Zone Mapping Implemented
   ✅ Quality Flags Added

📊 Overall Completion: 100.0%
🎉 Great job! You're ready for Day 2!


---

# 🚀 WHAT'S NEXT?

---

## 📅 DAY 2 PREVIEW: Data Quality & Cleaning Pipeline

Tomorrow you'll work on:
1. 🔍 Comprehensive data quality assessment
2. 🧹 Advanced cleaning procedures for IoT sensor data  
3. 📊 Missing data handling and interpolation strategies
4. 🚨 Outlier detection and treatment methods
5. 📏 Data standardization and normalization

## 📚 RECOMMENDED PREPARATION:
- Review PySpark DataFrame operations
- Read about time series data quality challenges
- Familiarize yourself with statistical outlier detection methods

## 💾 SAVE YOUR WORK:
- Commit your notebook to Git
- Document any issues or questions for tomorrow
- Save any custom functions you created

## 🤝 QUESTIONS?
- Post in the class discussion forum
- Review Spark documentation for any unclear concepts
- Prepare questions for tomorrow's Q&A session

In [ ]:
# TODO: Save your progress
print("\n💾 Don't forget to save your notebook and commit your changes!")

# Clean up (optional)
# spark.stop()